# Rethinking SZZ Search Strategies: Results

Reproduce the saved RQ1 and commit-message ablation results. DIO-V is included as a supplementary analysis at the end. Saved cell outputs can be viewed without running this notebook.

**Run all cells** to verify the packaged evidence and regenerate the results. This does not download subject repositories, execute logged commands, call a model, or use an API key. Calculations use the existing analysis scripts; this notebook only presents their outputs.

Setup from the package directory: `python -m pip install -r analysis/notebook-requirements.txt`, then `python -m jupyterlab analysis/results.ipynb`. Use Python 3.11+.

In [1]:
import csv
import html
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
from IPython.display import HTML, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'analysis/reproduce.py').is_file() and (p / 'results/manifest.json').is_file())
(ROOT / 'outputs').mkdir(exist_ok=True)
OUTPUT = Path(tempfile.mkdtemp(prefix='notebook-', dir=ROOT / 'outputs')) / 'analysis'
result = subprocess.run(
    [sys.executable, '-B', 'analysis/reproduce.py', '--numeric-only',
     '--sections', 'rq1', 'further_analysis', '--output', str(OUTPUT.relative_to(ROOT))],
    cwd=ROOT, env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
    capture_output=True, text=True,
)
if result.returncode:
    raise RuntimeError(result.stderr + result.stdout)
print(result.stdout.strip())
print('Generated files:', OUTPUT.relative_to(ROOT).as_posix())

def read_json(relative):
    return json.loads((OUTPUT / relative).read_text())

def read_table(relative):
    with (OUTPUT / relative).open(newline='') as stream:
        return {row['method']: row for row in csv.DictReader(stream)}

def number(value, digits=3):
    return f'{float(value):.{digits}f}'

def significance(p):
    return '**' if p < .01 else '*' if p < .05 else 'n.s.'

def table(headers, rows, caption):
    esc = lambda value: html.escape(str(value))
    head = '<tr>' + ''.join(f'<th>{esc(x)}</th>' for x in headers) + '</tr>'
    body = ''.join('<tr>' + ''.join(f'<td>{esc(x)}</td>' for x in row) + '</tr>' for row in rows)
    display(HTML(f'<table><caption>{esc(caption)}</caption><thead>{head}</thead><tbody>{body}</tbody></table>'))

validation = read_json('validation.json')
assert validation['model_calls'] == 0
assert validation['message_ablation']['primary_runs'] == 237
print('Evidence files verified:', validation['verified_files'])
print('Message-free runs: 237 primary; 1 user-interrupted attempt excluded.')

Verified and regenerated: rq1, further_analysis. No model calls.
Generated files: outputs/notebook-rkdmu2ln/analysis
Evidence files verified: 17293
Message-free runs: 237 primary; 1 user-interrupted attempt excluded.


## 1. RQ1: Accuracy and resource use

Accuracy covers all 79 cases; unresolved runs are scored as empty answers. For LLM- and agent-based methods, TP/FP/FN and precision/recall/F1 are averaged across three repetitions (237 slots). Rule-based methods have one evaluation per case. Time and API-equivalent cost are averaged over completed runs; their denominator is shown separately. Rule-based time is not reported and model API cost is zero.

In [2]:
rq1 = read_table('rq1/accuracy.csv')
resources = read_json('rq1/resources.json')['methods']
rules = ['B-SZZ', 'AG-SZZ', 'MA-SZZ', 'R-SZZ', 'L-SZZ', 'RA-SZZ*']
agents = ['LLM4SZZ', 'AgentSZZ', 'SZZ-Agent', 'Simple-SZZ-Agent', 'Claude Code']
rows = []
for method in rules + agents:
    row, resource = rq1[method], resources.get(method)
    rows.append([method, row['runs'], *[number(row[k], 2) for k in ('tp', 'fp', 'fn')],
                 *[number(row[k]) for k in ('precision', 'recall', 'f1')],
                 resource['normal_runs'] if resource else '—',
                 number(resource['mean_seconds'], 2) if resource else '—',
                 number(resource['mean_usd']) if resource else '0.000'])
table(['Method', 'Accuracy slots', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1',
       'Resource runs', 'Time (s)', 'API-equivalent USD'], rows,
      'RQ1: accuracy and mean resources per run')

Method,Accuracy slots,TP,FP,FN,Precision,Recall,F1,Resource runs,Time (s),API-equivalent USD
B-SZZ,79,52.00,134.00,27.00,0.280,0.658,0.392,—,—,0.000
AG-SZZ,79,47.00,133.00,32.00,0.261,0.595,0.363,—,—,0.000
MA-SZZ,79,45.00,141.00,34.00,0.242,0.570,0.340,—,—,0.000
R-SZZ,79,37.00,32.00,42.00,0.536,0.468,0.500,—,—,0.000
L-SZZ,79,33.00,36.00,46.00,0.478,0.418,0.446,—,—,0.000
RA-SZZ*,79,39.00,119.00,40.00,0.247,0.494,0.329,—,—,0.000
LLM4SZZ,237,46.67,28.33,32.33,0.622,0.591,0.606,237,51.91,0.090
AgentSZZ,237,50.67,18.33,28.33,0.735,0.641,0.685,237,33.32,0.137
SZZ-Agent,237,48.67,25.67,30.33,0.655,0.616,0.635,232,112.82,0.453
Simple-SZZ-Agent,237,55.00,24.00,24.00,0.696,0.696,0.696,237,80.99,0.266


### Paired F1 comparisons

Two-sided Wilcoxon signed-rank tests use 79 paired case-level F1 scores (three-run means for agent methods), not 237 independent observations. Holm correction covers the ten baseline comparisons. Positive differences mean higher F1 for Claude Code. `*`: adjusted p < 0.05; `**`: adjusted p < 0.01; `n.s.`: not significant.

In [3]:
rq1_tests = read_json('rq1/tests.json')
table(['Baseline', 'Cases', 'W', 'Raw p', 'Holm p', 'Significance'],
      [[method, t['cases'], number(t['statistic'], 1),
        f"{t['p_value']:.6g}", f"{t['p_holm']:.6g}", significance(t['p_holm'])]
       for method in rules + agents[:-1] for t in [rq1_tests[method]]],
      'RQ1: F1 differences from Claude Code')

Baseline,Cases,W,Raw p,Holm p,Significance
B-SZZ,79,103.0,1.15874e-05,6.95246e-05,**
AG-SZZ,79,65.5,4.73082e-06,3.31157e-05,**
MA-SZZ,79,60.0,1.93192e-06,1.54554e-05,**
R-SZZ,79,65.5,2.03181e-05,0.00010159,**
L-SZZ,79,24.5,5.31846e-07,4.78661e-06,**
RA-SZZ*,79,65.5,1.25704e-07,1.25704e-06,**
LLM4SZZ,79,25.0,0.000268121,0.00107248,**
AgentSZZ,79,22.0,0.00060582,0.00181746,**
SZZ-Agent,79,37.0,0.000632549,0.00181746,**
Simple-SZZ-Agent,79,38.0,0.0361816,0.0361816,*


## 2. Commit-message ablation

The RQ1 message-present runs are compared with the November 3, 2026 message-free experiment: 79 cases × 3 runs per condition. The original oracle is unchanged. ID 10/r02 uses the approved manual restart; the earlier user-interrupted attempt is excluded.

**Total time** is the outer attempt duration, including pre/postflight validation. **Process time** covers only the Claude process and is reported separately. Queue waiting and discarded attempts are excluded. Bash calls are tool calls, not individual shell commands. Cost uses the same token prices for both conditions and is not an actual subscription bill.

In [4]:
message_accuracy = read_table('further_analysis/message-accuracy.csv')
message_resources = read_json('further_analysis/message-resources.json')
rows = []
for condition, label in [('present', 'With messages'), ('free', 'Without messages')]:
    row = message_accuracy[condition]
    resource = message_resources[condition]['resources']
    rows.append([label, row['runs'], *[number(row[k], 2) for k in ('tp', 'fp', 'fn')],
                 *[number(row[k]) for k in ('precision', 'recall', 'f1')],
                 number(resource['bash_calls']['mean'], 2),
                 number(resource['total_seconds']['mean'], 2),
                 number(resource['common_cost_usd']['mean'])])
table(['Condition', 'Runs', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1',
       'Bash calls', 'Total time (s)', 'API-equivalent USD'], rows,
      'Commit-message ablation: repetition-mean accuracy and mean resources per run')

Condition,Runs,TP,FP,FN,Precision,Recall,F1,Bash calls,Total time (s),API-equivalent USD
With messages,237,61.33,18.67,17.67,0.767,0.776,0.771,6.80,50.14,0.166
Without messages,237,59.67,18.33,19.33,0.765,0.755,0.760,8.78,102.80,0.197


### Paired comparisons

Tests use 79 paired means, one per case and condition. Holm correction covers F1, Bash calls, total time, and cost. The tested case-mean F1 is different from the repetition-mean F1 in the descriptive table above. Positive differences mean larger values without messages. Non-significance does not establish equivalence.

In [5]:
message_tests = read_json('further_analysis/message-tests.json')
metrics = [('f1', 'Case-mean F1'), ('bash_calls', 'Bash calls'),
           ('total_seconds', 'Total time (s)'), ('common_cost_usd', 'API-equivalent USD')]
table(['Metric', 'With', 'Without', 'Difference', 'Raw p', 'Holm p', 'Significance'],
      [[label, number(t['mean_present'], 4), number(t['mean_free'], 4),
        number(t['mean_difference'], 4), f"{t['p_raw']:.6g}", f"{t['p_holm']:.6g}",
        significance(t['p_holm'])] for key, label in metrics for t in [message_tests[key]]],
      'Message ablation: paired tests (without minus with messages)')
sensitivity = read_json('further_analysis/message-sensitivity.json')
process = sensitivity['process_time_family']['process_seconds']
table(['Time boundary', 'With (s)', 'Without (s)', 'Holm p'],
      [['Claude process only', number(process['mean_present'], 2),
        number(process['mean_free'], 2), f"{process['p_holm']:.6g}"]],
      'Supplementary time comparison (separate four-test family)')

Metric,With,Without,Difference,Raw p,Holm p,Significance
Case-mean F1,0.7700,0.7511,-0.0190,0.339411,0.339411,n.s.
Bash calls,6.8017,8.7848,1.9831,1.69133e-05,4.67728e-05,**
Total time (s),50.1368,102.7972,52.6604,5.18044e-12,2.07218e-11,**
API-equivalent USD,0.1658,0.1965,0.0307,1.55909e-05,4.67728e-05,**


Time boundary,With (s),Without (s),Holm p
Claude process only,48.61,71.69,3.23814e-09


The two conditions ran at different calendar times and were not interleaved. Total time also includes validation work that differs between conditions; it is not a pure measure of model search time. The generated `message-sensitivity.json` additionally includes repository-equal weighting and exclusion of the two identity-header cases. See the [analysis notes](README.md#message-ablation) for the time and cost definitions.

## Supplement: DIO-V

DIO-V contains 67 cases after the documented corrections and exclusions. These results rescore the same saved answers, without new model runs. The dataset and all values in this section are unchanged by the message-ablation update.

In [6]:
dio = read_table('further_analysis/dio-v-accuracy.csv')
dio_tests = read_json('further_analysis/dio-v-tests.json')
table(['Method', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1', 'Holm p vs Claude Code'],
      [[method, *[number(dio[method][k], 2) for k in ('tp', 'fp', 'fn')],
        *[number(dio[method][k]) for k in ('precision', 'recall', 'f1')],
        f"{dio_tests[method]['p_holm']:.6g}" if method in dio_tests else '—'] for method in agents],
      'DIO-V: repetition-mean accuracy (67 cases; 201 slots per method)')
dio_validation = read_json('further_analysis/dio-v-validation.json')
print('Cases:', dio_validation['cases'])
print('Corrected labels:', ', '.join(dio_validation['corrected']))
print('Excluded cases:', ', '.join(dio_validation['excluded']))
print('Saved answers rescored:', dio_validation['rescored_answers'])

Method,TP,FP,FN,Precision,Recall,F1,Holm p vs Claude Code
LLM4SZZ,47.33,15.67,19.67,0.751,0.706,0.728,0.000354971
AgentSZZ,51.33,9.00,15.67,0.851,0.766,0.806,0.000354971
SZZ-Agent,50.00,15.00,17.00,0.769,0.746,0.758,0.000354971
Simple-SZZ-Agent,55.33,11.67,11.67,0.826,0.826,0.826,0.0047124
Claude Code,63.00,4.33,4.00,0.936,0.940,0.938,—


Cases: 67
Corrected labels: 907, 1381, 1728, 3530
Excluded cases: 313, 399, 717, 718, 975, 1121, 1470, 2357, 2513, 2557, 2911, 3521
Saved answers rescored: 1005
